# 04 - Data Warehousing on the Lakehouse (your technical spike)
Build a real star schema end-to-end with the medallion architecture:

```
 BRONZE (raw, append-only, dupes)  ->  SILVER (deduped, typed, validated)  ->  GOLD (star schema)
 dw_bronze_orders                      dw_silver_orders                       dim_date
 dw_bronze_customers                                                         dim_customer  (SCD2)
                                                                             dim_product   (SCD1)
                                                                             fact_sales    (grain: one order line)
```
At each step, practice explaining **the design decision** and **the Databricks feature** behind it.
Companion reading: `docs/data_warehousing_spike.md`.

In [ ]:
import os, sys
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, REPO_ROOT)

from pyspark.sql import functions as F
from spark_lab import datagen, techniques as T
from warehouse.delta_scd import scd2_merge_sql

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "interview_prep")
CATALOG, SCHEMA = dbutils.widgets.get("catalog"), dbutils.widgets.get("schema")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE {CATALOG}.{SCHEMA}")

def try_sql(stmt):
    """Some features depend on workspace tier/compute - show the error instead of failing the notebook."""
    try:
        return spark.sql(stmt)
    except Exception as e:
        print(f"[skipped] {type(e).__name__}: {str(e).splitlines()[0][:200]}")

## 1. Bronze: land raw data as-is (append-only, keep ingest metadata)

In [ ]:
datagen.raw_orders_with_dupes(spark, 1_000_000).write.mode("overwrite").saveAsTable("dw_bronze_orders")
datagen.customers(spark, 100_000).write.mode("overwrite").saveAsTable("dw_bronze_customers")
datagen.products(spark, 5_000).write.mode("overwrite").saveAsTable("dw_bronze_products")
display(spark.sql("SELECT count(*) AS rows, count(DISTINCT order_id) AS distinct_orders FROM dw_bronze_orders"))

## 2. Silver: dedupe, validate, conform
Duplicates come from at-least-once delivery / CDC replays. Keep the **latest** version per business key.

In [ ]:
bronze = spark.table("dw_bronze_orders")
silver = T.latest_per_key_window(bronze, "order_id", "ingest_ts")

checks = silver.agg(
    F.count("*").alias("rows"),
    F.sum(F.col("amount").isNull().cast("int")).alias("null_amount"),
    F.sum((F.col("amount") <= 0).cast("int")).alias("non_positive_amount"),
    F.sum(F.col("customer_id").isNull().cast("int")).alias("null_customer"),
)
display(checks)
silver.write.mode("overwrite").saveAsTable("dw_silver_orders")

# Enforced constraint: Delta rejects any future write that violates it.
try_sql("ALTER TABLE dw_silver_orders ADD CONSTRAINT positive_amount CHECK (amount > 0)")

**Talk track:** "In production I'd make this a Lakeflow Declarative Pipeline with expectations
(`EXPECT ... ON VIOLATION DROP ROW / FAIL UPDATE`) so data quality is declarative and tracked in the event log,
and use Auto Loader for incremental ingestion with schema evolution into bronze."

## 3. dim_date (generated, not loaded)

In [ ]:
%sql
CREATE OR REPLACE TABLE dim_date AS
SELECT
  CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_sk,   -- smart key: readable, sortable, cluster-friendly
  d                                        AS full_date,
  year(d) AS year, quarter(d) AS quarter, month(d) AS month,
  date_format(d, 'MMMM') AS month_name,
  weekofyear(d) AS week_of_year, dayofweek(d) AS day_of_week,
  dayofweek(d) IN (1, 7) AS is_weekend
FROM (SELECT explode(sequence(DATE'2024-01-01', DATE'2026-12-31', INTERVAL 1 DAY)) AS d);

ALTER TABLE dim_date ALTER COLUMN date_sk SET NOT NULL;

### Declare the primary key
This is an informational PRIMARY KEY: `RELY` lets the optimizer trust it. `try_sql` reports an error instead of failing if the feature isn't available on your tier.

In [ ]:
try_sql("ALTER TABLE dim_date ADD CONSTRAINT dim_date_pk PRIMARY KEY (date_sk) RELY")

## 4. dim_customer as SCD Type 2
- Surrogate key from an **IDENTITY** column (stable join key, decouples from source system IDs, handles history)
- `row_hash` for cheap change detection
- `valid_from` / `valid_to` / `is_current` for point-in-time joins

In [ ]:
%sql
CREATE OR REPLACE TABLE dim_customer (
  customer_sk  BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY,
  customer_id  BIGINT NOT NULL,          -- natural / business key
  name STRING, email STRING, region STRING, segment STRING,
  row_hash     STRING,
  valid_from   DATE NOT NULL,
  valid_to     DATE,                     -- NULL = still current
  is_current   BOOLEAN NOT NULL
) CLUSTER BY (customer_id);

### Generate and run the SCD2 MERGE (initial load)
`scd2_merge_sql` builds the staged-updates MERGE for the tracked columns, and the cell prints it so you can read it. The first load stages every bronze customer as of 2025-01-01. Against an empty table every row is "not matched", so all of them are inserted.

In [ ]:
TRACKED = ["name", "email", "region", "segment"]
merge_sql = scd2_merge_sql("dim_customer", "customer_changes", "customer_id", TRACKED)
print(merge_sql)

# Initial load = the same MERGE against an empty table (every row is "not matched").
(spark.table("dw_bronze_customers").withColumn("effective_date", F.lit("2025-01-01").cast("date"))
      .createOrReplaceTempView("customer_changes"))
display(spark.sql(merge_sql))

### Apply a change batch
On 2025-07-01, 5% of customers move to APAC, 2% become enterprise, and 500 new customers arrive. The MERGE output shows `num_updated_rows` (old versions closed) and `num_inserted_rows` (new versions plus new customers).

In [ ]:
# Change batch on 2025-07-01: 5% of customers move region, 2% change segment, 500 new customers arrive.
changes = (spark.table("dw_bronze_customers")
           .filter((F.col("customer_id") % 20 == 0) | (F.col("customer_id") % 50 == 1))
           .withColumn("region", F.when(F.col("customer_id") % 20 == 0, F.lit("APAC")).otherwise(F.col("region")))
           .withColumn("segment", F.when(F.col("customer_id") % 50 == 1, F.lit("enterprise")).otherwise(F.col("segment"))))
new_customers = datagen.customers(spark, 100_500).filter("customer_id >= 100000")
(changes.unionByName(new_customers).withColumn("effective_date", F.lit("2025-07-01").cast("date"))
        .createOrReplaceTempView("customer_changes"))

display(spark.sql(merge_sql))   # see num_updated_rows (closed versions) and num_inserted_rows

### Re-run the same batch
A safe load is **idempotent**: running the same batch again must change nothing. Expect 0 affected rows.

In [ ]:
# Idempotency check: re-running the same batch must change nothing.
display(spark.sql(merge_sql))

### Inspect the history
Look at a few customers: some that moved region (0, 20), one that changed segment (1), and a new one (100001). Changed customers have two versions with back-to-back validity windows.

In [ ]:
%sql
SELECT customer_id, customer_sk, region, segment, valid_from, valid_to, is_current
FROM dim_customer WHERE customer_id IN (0, 1, 20, 100001) ORDER BY customer_id, valid_from;

### The Delta transaction log
`DESCRIBE HISTORY` lists every commit (the CREATE and each MERGE) with its operation metrics. It's your audit trail, and the basis of time travel and `RESTORE`.

In [ ]:
%sql
-- Delta keeps every commit: audit + time travel (great for debugging a bad load and for rollback via RESTORE)
DESCRIBE HISTORY dim_customer;

### Time travel
Compare the row count at version 1 (just after the initial load) with the current count. The difference is the closed and new versions that the change batch added.

In [ ]:
%sql
SELECT 'version 1 (initial load)' AS v, count(*) FROM dim_customer VERSION AS OF 1
UNION ALL SELECT 'current', count(*) FROM dim_customer;

## 5. dim_product as SCD Type 1 (overwrite, no history)

In [ ]:
%sql
CREATE OR REPLACE TABLE dim_product (
  product_sk BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY,
  product_id BIGINT NOT NULL, product_name STRING, category STRING, list_price DECIMAL(10,2)
);

MERGE INTO dim_product t
USING dw_bronze_products s ON t.product_id = s.product_id
WHEN MATCHED AND (t.list_price <> s.list_price OR t.category <> s.category) THEN
  UPDATE SET t.product_name = s.product_name, t.category = s.category, t.list_price = s.list_price
WHEN NOT MATCHED THEN
  INSERT (product_id, product_name, category, list_price) VALUES (s.product_id, s.product_name, s.category, s.list_price);

## 6. fact_sales: point-in-time surrogate key lookup
Grain = one order line. Each sale must point at the customer version **valid on the order date**.

In [ ]:
%sql
CREATE OR REPLACE TABLE fact_sales
CLUSTER BY (date_sk, customer_sk) AS
SELECT
  o.order_id,
  CAST(date_format(o.order_date, 'yyyyMMdd') AS INT) AS date_sk,
  c.customer_sk,
  p.product_sk,
  o.quantity,
  CAST(o.amount AS DECIMAL(12,2)) AS amount,
  o.status
FROM dw_silver_orders o
JOIN dim_customer c
  ON  o.customer_id = c.customer_id
  AND o.order_date >= c.valid_from
  AND (c.valid_to IS NULL OR o.order_date < c.valid_to)    -- half-open interval [valid_from, valid_to)
JOIN dim_product p ON o.product_id = p.product_id;

### Reconcile the fact load
Every silver order should land in `fact_sales` exactly once. If the counts differ, the point-in-time join either fanned out (overlapping windows) or dropped rows (no matching version).

In [ ]:
# Reconciliation: every silver row landed exactly once (no fan-out from the range join, no drops).
display(spark.sql("""
  SELECT (SELECT count(*) FROM dw_silver_orders) AS silver_rows,
         (SELECT count(*) FROM fact_sales)       AS fact_rows,
         (SELECT count(DISTINCT order_id) FROM fact_sales) AS distinct_orders
"""))

### Add constraints and statistics
Add the PK/FK constraints that document the star schema, then run `ANALYZE TABLE` so the cost-based optimizer has row counts and column statistics. Each statement is wrapped in `try_sql`, so an unsupported feature is reported instead of stopping the notebook.

In [ ]:
for stmt in [
    "ALTER TABLE dim_customer ADD CONSTRAINT dim_customer_pk PRIMARY KEY (customer_sk) RELY",
    "ALTER TABLE dim_product ADD CONSTRAINT dim_product_pk PRIMARY KEY (product_sk) RELY",
    "ALTER TABLE fact_sales ADD CONSTRAINT fk_customer FOREIGN KEY (customer_sk) REFERENCES dim_customer",
    "ALTER TABLE fact_sales ADD CONSTRAINT fk_product FOREIGN KEY (product_sk) REFERENCES dim_product",
    "ALTER TABLE fact_sales ADD CONSTRAINT fk_date FOREIGN KEY (date_sk) REFERENCES dim_date",
    "ANALYZE TABLE fact_sales COMPUTE STATISTICS FOR ALL COLUMNS",
    "ANALYZE TABLE dim_customer COMPUTE STATISTICS FOR ALL COLUMNS",
]:
    try_sql(stmt)

**Talk track:** "PK/FK constraints in Unity Catalog are *informational* - not enforced - but with `RELY` the
optimizer can use them, e.g. to eliminate an unnecessary join or simplify DISTINCT. They also document the model
for BI tools and Genie. CHECK and NOT NULL *are* enforced. `ANALYZE` feeds the cost-based optimizer with row counts
and column stats so it picks the right join order and strategy; predictive optimization can run it for you."

## 7. Analytical queries you should be fluent in

In [ ]:
%sql
-- Why SCD2 matters: revenue by region as it WAS at sale time vs restated to today's region.
SELECT d.month,
       c_hist.region AS region_at_sale_time,
       c_now.region  AS region_today,
       sum(f.amount) AS revenue
FROM fact_sales f
JOIN dim_date d          ON f.date_sk = d.date_sk
JOIN dim_customer c_hist ON f.customer_sk = c_hist.customer_sk
JOIN dim_customer c_now  ON c_hist.customer_id = c_now.customer_id AND c_now.is_current
WHERE d.year = 2025 AND c_hist.region <> c_now.region
GROUP BY ALL ORDER BY d.month LIMIT 24;

### Subtotals with ROLLUP
`GROUP BY ROLLUP (quarter, category)` returns totals per quarter and category, subtotals per quarter, and a grand total, all in one pass. `grouping_id()` tells you which level each row is.

In [ ]:
%sql
-- Subtotals in one pass: ROLLUP / GROUPING SETS (classic BI cube queries)
SELECT d.quarter, p.category, sum(f.amount) AS revenue, grouping_id() AS level
FROM fact_sales f JOIN dim_date d USING (date_sk) JOIN dim_product p USING (product_sk)
GROUP BY ROLLUP (d.quarter, p.category)
ORDER BY d.quarter NULLS LAST, p.category NULLS LAST;

### Top N per group with QUALIFY
Rank customers by revenue within each region, then keep the top 3 with `QUALIFY`, which filters on a window function without a subquery.

In [ ]:
%sql
-- Top 3 customers per region with QUALIFY (filter on a window function without a subquery)
SELECT c.region, c.customer_id, sum(f.amount) AS revenue,
       rank() OVER (PARTITION BY c.region ORDER BY sum(f.amount) DESC) AS rnk
FROM fact_sales f JOIN dim_customer c USING (customer_sk)
GROUP BY c.region, c.customer_id
QUALIFY rnk <= 3
ORDER BY c.region, rnk;

### Running totals and month-over-month growth
Aggregate to months first, then use a window frame for year-to-date revenue and `lag` for growth versus the previous month.

In [ ]:
%sql
-- Running total + month-over-month growth (window frames, LAG)
WITH m AS (
  SELECT d.year, d.month, sum(f.amount) AS revenue
  FROM fact_sales f JOIN dim_date d USING (date_sk) GROUP BY ALL
)
SELECT *, sum(revenue) OVER (PARTITION BY year ORDER BY month ROWS UNBOUNDED PRECEDING) AS ytd_revenue,
       round(100 * (revenue / lag(revenue) OVER (ORDER BY year, month) - 1), 2) AS mom_growth_pct
FROM m ORDER BY year, month;

## 8. Gold serving layer: aggregates & semantic objects
Materialized views keep a pre-aggregated result incrementally refreshed (needs serverless / a SQL warehouse;
may be unavailable on some tiers - the cell reports it).

In [ ]:
try_sql("""
CREATE OR REPLACE MATERIALIZED VIEW gold_daily_sales AS
SELECT f.date_sk, c.region, p.category, sum(f.amount) AS revenue, count(*) AS order_lines
FROM fact_sales f JOIN dim_customer c USING (customer_sk) JOIN dim_product p USING (product_sk)
GROUP BY ALL
""")

# Plain view fallback: a governed semantic layer object, no storage.
spark.sql("""
CREATE OR REPLACE VIEW v_sales AS
SELECT f.*, d.full_date, d.year, d.month, c.customer_id, c.region, c.segment, p.category
FROM fact_sales f JOIN dim_date d USING (date_sk) JOIN dim_customer c USING (customer_sk) JOIN dim_product p USING (product_sk)
""")
spark.sql("COMMENT ON TABLE fact_sales IS 'Order-line grain sales fact. One row per order_id. Joins to dim_customer on point-in-time SK.'")
display(spark.sql("SELECT region, sum(amount) AS revenue FROM v_sales GROUP BY region ORDER BY revenue DESC"))

## 9. Stretch exercises (do these without looking anything up)
1. Add a **late-arriving change**: a region change effective 2025-04-01 that arrives *after* the 07-01 batch. What breaks in the MERGE? How does `AUTO CDC ... SEQUENCE BY` solve it?
2. Make the fact load **incremental**: only new `dw_silver_orders` since the last run (hint: Delta Change Data Feed, `table_changes()`, or a watermark column) and MERGE into `fact_sales`.
3. Handle an **early-arriving fact** (order for a customer not yet in dim_customer): insert an "inferred member" placeholder row and update it later.
4. Write the query that proves `dim_customer` has **no overlapping validity windows** per customer (hint: `lead(valid_from)` vs `valid_to`). *(This is merge_intervals!)*
5. Model a **many-to-many** (customer <-> marketing campaign) with a bridge table and explain double-counting risk.
6. Rewrite `fact_sales` for a **periodic snapshot** (daily inventory) vs this transactional fact - when is each right?